# Phase 8 — Model Experiments & Selection

## Objective

The objective of this phase is to train, evaluate, and compare multiple machine learning models for customer churn prediction.

Several baseline and ensemble learning algorithms will be trained using the feature-engineered dataset. Each model will be evaluated using a common set of classification metrics to ensure a fair comparison.

The best-performing models will be shortlisted for hyperparameter tuning in the next phase.

## Workflow

This phase is organized into the following sections:

1. Import Required Libraries
2. Load Feature-Engineered Dataset
3. Prepare Features and Target
4. Perform Stratified Train-Test Split
5. Train Baseline Models
6. Train Ensemble Models
7. Evaluate Model Performance
8. Compare Models
9. Select Top Models

## Part A — Import Required Libraries

The following libraries are used for data manipulation, model development, and performance evaluation.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

# Baseline Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Ensemble Models
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

## Part B — Load Feature-Engineered Dataset

The feature-engineered dataset created during the previous phase is loaded for model development.

This dataset contains:

- Cleaned observations
- Encoded categorical variables
- Scaled numerical features
- Engineered features
- Target variable

In [3]:
def load_dataset(file_path):
    """
    Load the feature-engineered dataset.
    """

    df = pd.read_csv(file_path)

    print("=" * 60)
    print("DATASET LOADED")
    print("=" * 60)
    print(f"Shape : {df.shape}")

    return df

In [4]:
df = load_dataset(
    "../data/processed/v1_feature_engineered_customer_churn.csv"
)

df.head()

DATASET LOADED
Shape : (7032, 32)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TotalServices,Churn
0,0,0,1,0,-1.280248,0,1,-1.161694,-0.994194,1,...,0,0,0,0,0,0,1,0,1,0
1,1,0,0,0,0.064303,1,0,-0.260878,-0.173740,0,...,0,0,0,1,0,0,0,1,3,0
2,1,0,0,0,-1.239504,1,1,-0.363923,-0.959649,0,...,0,0,0,0,0,0,0,1,3,1
3,1,0,0,0,0.512486,0,0,-0.747850,-0.195248,1,...,0,0,0,1,0,0,0,0,3,0
4,0,0,0,0,-1.239504,1,1,0.196178,-0.940457,0,...,0,0,0,0,0,0,1,0,2,1


## Part C — Prepare Features and Target

The dataset is divided into:

- Feature matrix (`X`)
- Target variable (`y`)

The target variable represents whether a customer churned.

In [5]:
def prepare_features_target(df, target="Churn"):
    """
    Separate feature matrix and target variable.
    """

    X = df.drop(columns=target)
    y = df[target]

    print("=" * 60)
    print("FEATURE MATRIX")
    print("=" * 60)
    print(f"Features : {X.shape}")
    print(f"Target   : {y.shape}")

    return X, y

In [6]:
X, y = prepare_features_target(df)

FEATURE MATRIX
Features : (7032, 31)
Target   : (7032,)


## Part D — Train-Test Split

The dataset is divided into training and testing subsets using stratified sampling.

Stratification preserves the original distribution of the target classes in both subsets, ensuring a fair evaluation of model performance.

In [7]:
def split_dataset(
    X,
    y,
    test_size=0.20,
    random_state=42
):
    """
    Perform a stratified train-test split.
    """

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    print("=" * 60)
    print("TRAIN-TEST SPLIT")
    print("=" * 60)
    print(f"Training Samples : {len(X_train)}")
    print(f"Testing Samples  : {len(X_test)}")

    return X_train, X_test, y_train, y_test

In [8]:
X_train, X_test, y_train, y_test = split_dataset(
    X,
    y
)

TRAIN-TEST SPLIT
Training Samples : 5625
Testing Samples  : 1407


## Part E — Train Baseline Models

This section trains multiple baseline classification models using the training dataset.

Each model is evaluated using the same performance metrics to ensure a fair comparison.

The following baseline models are included:

- Logistic Regression
- Decision Tree
- Gaussian Naive Bayes
- K-Nearest Neighbors
- Support Vector Machine

In [13]:
# Step 1 — Create a Generic Evaluation Function

def evaluate_model(model, X_train, X_test, y_train, y_test):
    """
    Train and evaluate a classification model.
    """

    # Train
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Probabilities (if supported)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = model.decision_function(X_test)
    else:
        y_prob = None

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        pos_label=1
    )

    recall = recall_score(
        y_test,
        y_pred,
        pos_label=1
    )

    f1 = f1_score(
        y_test,
        y_pred,
        pos_label=1
    )

    if y_prob is not None:
        roc_auc = roc_auc_score(
            y_test,
            y_prob
        )
    else:
        roc_auc = np.nan

    # Metrics
    # accuracy = accuracy_score(y_test, y_pred)
    # precision = precision_score(y_test, y_pred, pos_label="Yes")
    # recall = recall_score(y_test, y_pred, pos_label="Yes")
    # f1 = f1_score(y_test, y_pred, pos_label="Yes")

    # if y_prob is not None:
    #     roc_auc = roc_auc_score(
    #         y_test.map({"No": 0, "Yes": 1}),
    #         y_prob
    #     )
    # else:
    #     roc_auc = np.nan

    return {
        "Model": model.__class__.__name__,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": roc_auc,
        "Model Object": model
    }

In [14]:
# Step 2 — Create Baseline Models

baseline_models = {
    "Logistic Regression": LogisticRegression(random_state=42),

    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "Gaussian Naive Bayes": GaussianNB(),

    "K-Nearest Neighbors": KNeighborsClassifier(),

    "Support Vector Machine": SVC(
        probability=True,
        random_state=42
    )
}

In [15]:
# Step 3 — Train All Baseline Models

baseline_results = []

for name, model in baseline_models.items():

    print(f"Training {name}...")

    result = evaluate_model(
        model,
        X_train,
        X_test,
        y_train,
        y_test
    )

    baseline_results.append(result)

Training Logistic Regression...
Training Decision Tree...
Training Gaussian Naive Bayes...
Training K-Nearest Neighbors...
Training Support Vector Machine...


In [16]:
# Step 4 — Create Results Table

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df = baseline_results_df.drop(
    columns="Model Object"
)

baseline_results_df = baseline_results_df.sort_values(
    by="ROC-AUC",
    ascending=False
).reset_index(drop=True)

baseline_results_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,LogisticRegression,0.804549,0.649547,0.574866,0.609929,0.836022
1,GaussianNB,0.647477,0.420779,0.866310,0.566434,0.809890
2,SVC,0.788913,0.644195,0.459893,0.536661,0.801931
3,KNeighborsClassifier,0.764037,0.555263,0.564171,0.559682,0.779774
4,DecisionTreeClassifier,0.709311,0.456576,0.491979,0.473616,0.639741


### Observation

Five baseline classification models were trained and evaluated using the same train-test split.

#### Key Findings

- **Logistic Regression** achieved the best overall performance among the baseline models, with the highest Accuracy, F1-Score, and ROC-AUC.
- **Gaussian Naive Bayes** achieved the highest Recall, indicating its ability to identify most churning customers, although at the expense of lower Precision.
- **Support Vector Machine** demonstrated strong Precision but lower Recall, making it more conservative in predicting churn.
- **K-Nearest Neighbors** provided balanced performance but did not outperform Logistic Regression.
- **Decision Tree** produced the weakest overall performance, suggesting that a single tree may not generalize well for this dataset.

Based on these results, Logistic Regression serves as a strong baseline model for comparison with ensemble learning algorithms.

## Part F — Train Ensemble Models

Ensemble learning combines the predictions of multiple models to improve overall performance and generalization.

Tree-based ensemble methods are widely used for structured datasets because they can capture complex, non-linear relationships while reducing overfitting.

The following ensemble models are evaluated:

- Random Forest
- Extra Trees
- Gradient Boosting
- AdaBoost

In [17]:
# Step 1 — Create Ensemble Models

ensemble_models = {
    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "Extra Trees": ExtraTreesClassifier(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        random_state=42
    )
}

In [18]:
# Step 2 — Train Ensemble Models

ensemble_results = []

for name, model in ensemble_models.items():

    print(f"Training {name}...")

    result = evaluate_model(
        model,
        X_train,
        X_test,
        y_train,
        y_test
    )

    ensemble_results.append(result)

Training Random Forest...
Training Extra Trees...
Training Gradient Boosting...
Training AdaBoost...


In [19]:
# Step 3 — Create Results Table

ensemble_results_df = pd.DataFrame(ensemble_results)

ensemble_results_df = ensemble_results_df.drop(
    columns="Model Object"
)

ensemble_results_df = ensemble_results_df.sort_values(
    by="ROC-AUC",
    ascending=False
).reset_index(drop=True)

ensemble_results_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,AdaBoostClassifier,0.795309,0.648276,0.502674,0.566265,0.840830
1,GradientBoostingClassifier,0.793888,0.635484,0.526738,0.576023,0.840674
2,RandomForestClassifier,0.786070,0.620462,0.502674,0.555391,0.814211
3,ExtraTreesClassifier,0.774698,0.594059,0.481283,0.531758,0.791122


In [20]:
all_results = pd.concat(
    [baseline_results_df, ensemble_results_df],
    ignore_index=True
)

all_results = all_results.sort_values(
    by=["ROC-AUC", "F1-Score"],
    ascending=False
).reset_index(drop=True)

all_results

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,AdaBoostClassifier,0.795309,0.648276,0.502674,0.566265,0.840830
1,GradientBoostingClassifier,0.793888,0.635484,0.526738,0.576023,0.840674
2,LogisticRegression,0.804549,0.649547,0.574866,0.609929,0.836022
3,RandomForestClassifier,0.786070,0.620462,0.502674,0.555391,0.814211
4,GaussianNB,0.647477,0.420779,0.866310,0.566434,0.809890
5,SVC,0.788913,0.644195,0.459893,0.536661,0.801931
6,ExtraTreesClassifier,0.774698,0.594059,0.481283,0.531758,0.791122
7,KNeighborsClassifier,0.764037,0.555263,0.564171,0.559682,0.779774
8,DecisionTreeClassifier,0.709311,0.456576,0.491979,0.473616,0.639741


In [21]:
comparison_df = all_results.copy()

comparison_df["Accuracy"] = comparison_df["Accuracy"].round(4)
comparison_df["Precision"] = comparison_df["Precision"].round(4)
comparison_df["Recall"] = comparison_df["Recall"].round(4)
comparison_df["F1-Score"] = comparison_df["F1-Score"].round(4)
comparison_df["ROC-AUC"] = comparison_df["ROC-AUC"].round(4)

comparison_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,AdaBoostClassifier,0.7953,0.6483,0.5027,0.5663,0.8408
1,GradientBoostingClassifier,0.7939,0.6355,0.5267,0.5760,0.8407
2,LogisticRegression,0.8045,0.6495,0.5749,0.6099,0.8360
3,RandomForestClassifier,0.7861,0.6205,0.5027,0.5554,0.8142
4,GaussianNB,0.6475,0.4208,0.8663,0.5664,0.8099
5,SVC,0.7889,0.6442,0.4599,0.5367,0.8019
6,ExtraTreesClassifier,0.7747,0.5941,0.4813,0.5318,0.7911
7,KNeighborsClassifier,0.7640,0.5553,0.5642,0.5597,0.7798
8,DecisionTreeClassifier,0.7093,0.4566,0.4920,0.4736,0.6397


## Observation

Nine machine learning models were trained and evaluated using identical training and testing datasets.

### Key Findings

- **AdaBoost** achieved the highest ROC-AUC, indicating excellent discrimination between churned and retained customers.
- **Gradient Boosting** achieved the highest F1-Score among the ensemble models, demonstrating the best balance between Precision and Recall.
- **Logistic Regression** remained highly competitive, outperforming several more complex algorithms despite being the simplest model.
- **Random Forest** performed well but did not surpass Gradient Boosting or AdaBoost.
- **Decision Tree** produced the weakest overall performance, highlighting the benefit of ensemble learning over a single decision tree.

Overall, the ensemble methods demonstrated improved predictive performance, with **AdaBoost**, **Gradient Boosting**, and **Logistic Regression** emerging as the strongest candidates for further optimization.